# 원하는 포즈로 이미지 만들기 — 실습 초안

참조 사진 → OpenPose 관절 추출 → ControlNet 조건 → 이미지 생성 → 세 가지 비교 실험. 이 노트는 아직 Colab 실행 검증 전입니다. 실제 실행 결과와 관찰을 기록한 뒤 제출하세요.

기존 FLUX.2-klein/ComfyUI 노트에서 출발했지만 SD 1.5용 OpenPose ControlNet은 FLUX.2-klein에 바로 연결할 수 없어 별도 호환 조합인 Stable Diffusion 1.5 + OpenPose ControlNet을 사용합니다. 원문은 비슷한 모델을 허용합니다. 생성 결과와 모델 사용 조건을 직접 확인하세요.

준비: 런타임 → 런타임 유형 변경 → GPU. 사용 권리가 있는 전신 포즈 사진 2장. 모델 다운로드 및 GPU 이용은 인터넷과 Colab 상황에 따라 달라집니다. 기존 FLUX 서버와 GPU 메모리가 충돌하지 않도록 새 세션에서 이 노트만 실행하세요.

모델: [SD 1.5 미러](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5), [OpenPose ControlNet 모델 카드 및 예제](https://huggingface.co/lllyasviel/control_v11p_sd15_openpose). CreativeML OpenRAIL-M 사용 조건을 확인하세요.

## 1. GPU와 라이브러리 준비
GPU 사용 가능 여부를 확인하고 실습 라이브러리를 설치합니다. 설치 뒤 import 문제가 발생하면 세션을 다시 시작하고 순서대로 실행하세요.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 없습니다. 런타임 유형을 GPU로 변경하세요.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('PyTorch:', torch.__version__)

In [ ]:
# Colab 환경에서 사용할 버전을 지정해 설치합니다. 다운로드에 시간이 걸릴 수 있어요.
%pip -q install 'diffusers==0.30.3' 'transformers==4.44.2' 'accelerate==0.34.2' 'controlnet-aux==0.0.9' 'huggingface-hub>=0.23.2,<1.0' safetensors

## 2. 포즈 사진 두 장 업로드
파일 선택 창에서 참조 이미지 두 장을 함께 선택하세요. 사진은 본인이 공개할 수 있는 것을 사용합니다. 뒤의 실험에 A와 B로 복사하고 출처를 따로 기록하세요. 업로드 파일은 Colab 런타임에 저장됩니다.

In [ ]:
from google.colab import files
from pathlib import Path
from PIL import Image, ImageOps
from IPython.display import display

uploaded = files.upload()
names = list(uploaded)
if len(names) != 2:
    raise ValueError('포즈 사진을 정확히 두 장 선택하세요. 이 셀을 다시 실행하면 됩니다.')
out = Path('/content/pose_assignment')
samples = out / 'samples'
samples.mkdir(parents=True, exist_ok=True)

def prepare(path):
    # 512×512 안에 비율을 유지한 채 맞춥니다. 이미지가 잘리지 않아요.
    image = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    return ImageOps.pad(image, (512, 512), method=Image.Resampling.LANCZOS, color='white')

pose_a, pose_b = [prepare(name) for name in names]
pose_a.save(samples / 'pose_A.png')
pose_b.save(samples / 'pose_B.png')
print('A:', names[0], '/ B:', names[1])
display(pose_a, pose_b)

## 3. 사진에서 관절 추출
OpenPose 전처리기가 사진에서 관절을 추출해 ControlNet 입력용 뼈대 이미지를 만듭니다. 검은 화면만 나오거나 포즈가 빠졌다면 다른 사진으로 다시 시험하세요. 여러 인물이 있는 이미지에는 여러 포즈가 추출될 수 있습니다.

In [ ]:
from controlnet_aux import OpenposeDetector

detector = OpenposeDetector.from_pretrained('lllyasviel/ControlNet')
skeleton_a = detector(pose_a).resize((512, 512))
skeleton_b = detector(pose_b).resize((512, 512))
skeleton_a.save(samples / 'skeleton_A.png')
skeleton_b.save(samples / 'skeleton_B.png')
print('A와 B의 추출된 관절을 확인하세요:')
display(skeleton_a, skeleton_b)
del detector

## 4. 생성 모델과 포즈 ControlNet 불러오기
ControlNet 모델 카드에서 권장하는 SD 1.5 계열을 이용합니다. GPU 메모리를 아끼기 위해 CPU 오프로딩을 사용합니다. 첫 실행에는 여러 GB 다운로드가 필요할 수 있어요.

In [ ]:
import torch
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler

base_model = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
control_model = 'lllyasviel/control_v11p_sd15_openpose'
controlnet = ControlNetModel.from_pretrained(control_model, torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    base_model, controlnet=controlnet, torch_dtype=torch.float16
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
print('모델 준비 완료')

## 5. 세 가지 조건으로 생성
A1과 A2는 같은 포즈에 다른 프롬프트, A1과 B1은 같은 프롬프트에 다른 포즈입니다. 세 결과의 시드·스텝·ControlNet 강도는 같게 유지합니다. 포즈 강도가 높아도 정확히 따라 한다는 보장은 없습니다.

In [ ]:
import json
from datetime import datetime, timezone

prompt_1 = 'a single dancer in casual clothes in a bright dance studio, full body, photograph'
prompt_2 = 'a single astronaut in a space station, full body, cinematic illustration'
seed = 42
steps = 25
control_strength = 1.0
experiments = [('A1', skeleton_a, prompt_1), ('A2', skeleton_a, prompt_2), ('B1', skeleton_b, prompt_1)]
records = []
for label, skeleton, prompt in experiments:
    # 매번 같은 시드의 새 생성기를 만들어 비교합니다.
    generator = torch.Generator(device='cpu').manual_seed(seed)
    result = pipe(prompt=prompt, image=skeleton, generator=generator,
                  num_inference_steps=steps, controlnet_conditioning_scale=control_strength,
                  guidance_scale=7.5).images[0]
    target = samples / f'result_{label}.png'
    result.save(target)
    records.append({'id': label, 'prompt': prompt, 'seed': seed, 'steps': steps,
                    'guidance_scale': 7.5, 'control_strength': control_strength,
                    'base_model': base_model, 'control_model': control_model,
                    'output': str(target.relative_to(out))})
    print(label, '저장:', target)
    display(result)
(out / 'experiment_settings.json').write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
print('생성 설정 저장 완료')

## 6. 결과 다운로드
참조 사진, 뼈대, 결과 이미지와 실험 설정을 ZIP으로 내려받습니다. 로컬 `openpose` 폴더에 압축을 풀어 결과를 확인하세요. 출처와 공개 권리를 확인한 사진만 GitHub에 올립니다. 노트 자체는 Colab의 파일 → 다운로드 → .ipynb 다운로드를 이용하세요.

In [ ]:
import shutil
archive = shutil.make_archive('/content/pose_assignment_results', 'zip', out)
files.download(archive)

## 7. 실험 관찰 — 실행한 뒤 직접 작성

- A1 ↔ A2 (같은 포즈, 다른 프롬프트): **실제 결과를 보고 작성**
- A1 ↔ B1 (다른 포즈, 같은 프롬프트): **실제 결과를 보고 작성**
- 관절 추출이 잘되거나 실패한 부분: **실제 결과를 보고 작성**
- 손·얼굴·포즈의 한계와 개선할 점: **실제 결과를 보고 작성**
- 사진 출처와 공개 가능 여부: **직접 확인 후 작성**
- 실행 환경(GPU, 라이브러리 버전)과 실행 날짜: **직접 확인 후 작성**

이 초안은 결과가 없으므로 그대로 제출할 수 없습니다.